In [3]:
#befine the llm 
from langchain_core.messages import AIMessage, HumanMessage

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="gemma2-9b-it",
    temperature=0,
    max_tokens=None,
    timeout=60,
    groq_api_key="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",
    # other params...
)

llm.invoke([HumanMessage(content="hello how are you!")])


AIMessage(content="As an AI, I don't have feelings, but I'm here and ready to help! How can I assist you today? 😊\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 14, 'total_tokens': 46, 'completion_time': 0.058181818, 'prompt_time': 0.001901294, 'queue_time': 0.23557507, 'total_time': 0.060083112}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-34211779-b3d6-4e0b-9e05-1688863fd6fb-0', usage_metadata={'input_tokens': 14, 'output_tokens': 32, 'total_tokens': 46})

In [ ]:
# define the state 

from typing import Annotated
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage


class Store_message(TypedDict):
    """State to store messages."""

    messages: Annotated[list[AnyMessage], add_messages]
    resume_content: str = ""
    job_description: str =""
    url:str=""
    md_formated: str =""
    score: float = 0.0
    valid: bool = True

In [ ]:
import requests
from bs4 import BeautifulSoup
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage




path =r"D:\Help_project_list\resume_builder\reume_builder_checking\ai-resume-creator\Kevin_Andrews_Resume.pdf"

# Node: Load PDF and extract text and store in state
def load_pdf(state: StoreMessage) -> StoreMessage:
    from langchain_community.document_loaders import PyMuPDFLoader
    loader = PyMuPDFLoader(path)
    docs = loader.load()
    for i in docs:
        state["resume_content"]+=i.page_content
    return state


# Node: Extract job description
def get_job_description(state: StoreMessage) -> StoreMessage:
    response = requests.get(state["url"])
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        # You might need to adjust the selector based on the actual structure
        job_description = soup.get_text(separator="\n", strip=True)
        return  state["job_description"] = job_description
    else:
        raise Exception(f"Failed to fetch URL: {response.status_code}")


# Node: LLM with scoring
def llm_with_score(state: StoreMessage) -> StoreMessage:
    sys_msg = SystemMessage(content="You are an expert resume evaluator. You will assess how well a resume matches a specific job description and return a score from 0 to 100. Only return the score as a number. Do not include explanations or any extra text.")
    user_msg = HumanMessage(content=f"""
    {state["resume_content"]}

    {state["job_description"]}

    Give the match score out of 100. Only return the number.
""")
    final = llm.invoke([sys_msg, user_msg])
    # Dummy scoring logic
    state["score"] = int(final)  # This would be from your LLM comparison output
    return state



# Node: Conditionally create resume
def create_resume(state: StoreMessage) -> StoreMessage:
    sys_msg = SystemMessage(content="You are an expert resume writer and ATS (Applicant Tracking System) specialist. You create exceptional, high-quality resumes tailored to specific job descriptions and optimized to bypass ATS filters. Below, I will provide the original resume and the job description. Your task is to analyze them and generate an improved version of the resume that aligns closely with the job description. Make the content powerful, professional, and results-driven. Return the final improved resume in clean, well-formatted Markdown.")
    user_msg = HumanMessage(content=f"""
    my resume 
    {state["resume_content"]}
    job description
    {state["job_description"]}
    """)
    final = llm.invoke([sys_msg, user_msg])
    # Dummy Markdown resume creation
    state["resume_md"] = final.content
    return state


# Node: Review resume
def llm_review(state: StoreMessage) -> StoreMessage:
    state["valid"] = False
    # Append review comment or just return state
    return state


# Add edges
graph.add_edge(START, "load_pdf")
graph.add_edge("load_pdf", "get_job_description")
graph.add_edge("get_job_description", "llm_with_score")

# Add conditional edge
def score_check(state: StoreMessage) -> str:
    return "create_resume" if state["score"] > 70 else "llm_review"

graph.add_conditional_edges("llm_with_score", score_check)

graph.add_edge("create_resume", END)
graph.add_edge("llm_review", END)

# Compile the graph
workflow = graph.compile()

In [8]:
#gpt code 

# === Imports ===
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END, START

import requests
from bs4 import BeautifulSoup

# === Define State ===
class StoreMessage(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    resume_content: str
    job_description: str
    url: str
    md_formated: str
    score: float
    valid: bool

# === Import your LLM instance ===
# Define or import your LLM here (like from LangChain)
# For example:
paths = r"c:\Users\basilahamed.h\Downloads\DOC-20241108-WA0011. (3).pdf"
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="gemma2-9b-it",
    temperature=0,
    max_tokens=None,
    timeout=60,
    groq_api_key="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",
    # other params...
)

# === Define Nodes ===
def load_pdf(state: StoreMessage) -> StoreMessage:
    from langchain_community.document_loaders import PyMuPDFLoader
    path = paths
    # print("//// pdf path ")
    # print(path)
    # print("//// pdf path ")
    loader = PyMuPDFLoader(path)
    docs = loader.load()
    for doc in docs:
        # print(doc.page_content)
        state["resume_content"] += doc.page_content
    return state

def get_job_description(state: StoreMessage) -> StoreMessage:
    response = requests.get(state["url"])
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        job_description = soup.get_text(separator="\n", strip=True)
        state["job_description"] = job_description
        return state
    else:
        raise Exception(f"Failed to fetch URL: {response.status_code}")

def llm_with_score(state: StoreMessage) -> StoreMessage:
    sys_msg = SystemMessage(content="You are an expert resume evaluator. Assess the resume's match to the job description and return a score from 0 to 100.")
    user_msg = HumanMessage(content=f"""
{state['resume_content']}

Job Description:
{state['job_description']}

Give the match score out of 100. Only return the number.
""")
    result = llm.invoke([sys_msg, user_msg])
    try:
        state["score"] = float(result.content.strip())
    except ValueError:
        state["score"] = 0.0
    return state

def create_resume(state: StoreMessage) -> StoreMessage:
    sys_msg = SystemMessage(content="""
    
    You are a professional resume writer specializing in creating ATS-friendly resumes. Your task is to create a well-formatted resume in Markdown format based on the information provided by the user.

Follow these strict formatting rules:
1. Use a single # for the person's full name at the top
2. Put contact information on one line with | separators
3. Use ## for main section headings (Summary, Technical Skills, Professional Experience, etc.)
4. Use ### for job titles, degree names, or project names
5. Use **Bold** for companies, skill categories, and other emphasis
6. Use - for bullet points in lists
7. Format job history with company name and dates on the same line
8. Include quantifiable achievements in bullet points
9. Structure the resume in this exact order: Name, Contact Info, Summary, Technical Skills, Professional Experience, Education, Projects, Certifications, Languages/Additional Info

Your output must follow this exact Markdown structure:
```markdown
# [Full Name]
[Location] | [Email] | [Phone] | [LinkedIn]

## Summary
[Concise professional summary highlighting expertise, years of experience, and key strengths]

## Technical Skills
- **[Category]**: [Skill 1], [Skill 2], [Skill 3]
- **[Category]**: [Skill 1], [Skill 2], [Skill 3]
[Continue for all relevant technical skill categories]

## Professional Experience

### [Job Title]
**[Company Name]** | [Start Date] - [End Date]
- [Achievement with quantifiable result]
- [Achievement with quantifiable result]
- [Achievement with quantifiable result]
- [Achievement with quantifiable result]

### [Previous Job Title]
**[Previous Company Name]** | [Start Date] - [End Date]
- [Achievement with quantifiable result]
- [Achievement with quantifiable result]
- [Achievement with quantifiable result]
- [Achievement with quantifiable result]

## Education
### [Degree]
**[University Name]** | [Year Started] - [Year Graduated]
- [GPA if applicable]
- [Relevant coursework or achievements]

## Projects
### [Project Name]
- [Description with technologies used]
- [Key feature or achievement]
- [Deployment or result]

### [Another Project Name]
- [Description with technologies used]
- [Key feature or achievement]
- [Deployment or result]

## Certifications
- [Certification 1]
- [Certification 2]
- [Certification 3]

## Languages
- [Language 1] ([Proficiency Level])
- [Language 2] ([Proficiency Level])
    """)
    user_msg = HumanMessage(content=f"""
My resume:
{state["resume_content"]}

Job description:
{state["job_description"]}
""")
    final = llm.invoke([sys_msg, user_msg])
    state["md_formated"] = final.content
    return state

def llm_review(state: StoreMessage) -> StoreMessage:
    state["valid"] = False
    return state

# === Graph Definition ===
graph = StateGraph(StoreMessage)

graph.add_node("load_pdf", load_pdf)
graph.add_node("get_job_description", get_job_description)
graph.add_node("llm_with_score", llm_with_score)
graph.add_node("create_resume", create_resume)
graph.add_node("llm_review", llm_review)

graph.set_entry_point("load_pdf")

graph.add_edge("load_pdf", "get_job_description")
graph.add_edge("get_job_description", "llm_with_score")

def score_check(state: StoreMessage) -> str:
    return "create_resume" if state["score"] > 70 else "llm_review"

graph.add_conditional_edges("llm_with_score", score_check)

graph.add_edge("create_resume", END)
graph.add_edge("llm_review", END)

workflow = graph.compile()


In [9]:
initial_state = {
    "messages": [],
    "resume_content": "",
    "job_description": "",
    "url": "https://jobs.six-group.com/job/Warsaw-%28Senior%29-FrontFull-stack-Developer/1193235301/",  # replace this with a real job posting URL
    "md_formated": "",
    "score": 0.0,
    "valid": True,
}
messages = workflow.invoke(initial_state)
for m in messages['messages']:
    m.pretty_print()




final_state = workflow.invoke(initial_state)

In [2]:
import os

def save_markdown_to_file(md_content: str, file_path: str = "optimized_resume.md") -> str:
    # Get the absolute path of the file
    full_path = os.path.abspath(file_path)
    
    # Write the content to the file
    with open(full_path, "w", encoding="utf-8") as f:
        f.write(md_content)
    
    print(f"✅ Markdown resume saved to: {full_path}")
    return full_path

In [10]:
final_state = workflow.invoke({
    "messages": [],
    "resume_content": "",
    "job_description": "",
    "url": "https://jobs.six-group.com/job/Warsaw-%28Senior%29-FrontFull-stack-Developer/1193235301/",
    "md_formated": "",
    "score": 0.0,
    "valid": True
})

# Save the markdown file

if len(final_state["md_formated"]) > 0:
    final_path  = save_markdown_to_file(final_state["md_formated"])
    save_padf = pdf_convector(final_path)
else:
    print("NOT HAVE THE MARKDOWN FILE")


NOT HAVE THE MARKDOWN FILE


In [5]:
final_state

{'messages': [],
 'resume_content': 'Kevin Andrews \nFull Stack Developer \nHighly \nmotivated \nfull-stack \ndeveloper \nwith\nexpertise in front-end and back-end technologies,\nfocused on building eﬃcient, scalable, and user-\nfriendly web applications. Skilled in problem-\nsolving, clean code, and continuous learning, with\na strong attention to detail and a collaborative,\nsolution-driven approach. \nkevinandrews001@gmail.com \n09994053302 \nThiruvarur, India \nkevinandrews-portfolio.netlify.app/ \nlinkedin.com/in/kevinandrewsv \ngithub.com/Kevinandrewsv \nEDUCATION \nFull Stack Development \nGUVI Geek Network, IITM Research Park \n05/2023 - 05/2024,  \nChennai \nBachelor of Engineering, Mechanical \nJayam college of Engineering and\nTechnology \n05/2013 - 04/2017,  \nDharmapuri \nPERSONAL PROJECTS \nDoctor Appointment Booking Website\n (05/2024 - 06/2024) \nTechnologies used : React.js, Nodejs / Express.js, MongoDB on\nAWS \nDescription: Developed a web application for booking,\nm

In [11]:
final_state = workflow.invoke({
    "messages": [],
    "resume_content": "",
    "job_description": "",
    "url": "https://www.zoho.com/careers/jobdetails/?job_id=2803000614929615",
    "md_formated": "",
    "score": 0.0,
    "valid": True
})

# Save the markdown file

if len(final_state["md_formated"]) > 0:
    final_path  = save_markdown_to_file(final_state["md_formated"])
    save_padf = pdf_convector(final_path)
else:
    print("resume not satisfied.")


✅ Markdown resume saved to: d:\Help_project_list\resume_builder\reume_builder_checking\ai-based-resume-builder-streamlit\optimized_resume.md


In [7]:
final_state

{'messages': [],
 'resume_content': 'Kevin Andrews \nFull Stack Developer \nHighly \nmotivated \nfull-stack \ndeveloper \nwith\nexpertise in front-end and back-end technologies,\nfocused on building eﬃcient, scalable, and user-\nfriendly web applications. Skilled in problem-\nsolving, clean code, and continuous learning, with\na strong attention to detail and a collaborative,\nsolution-driven approach. \nkevinandrews001@gmail.com \n09994053302 \nThiruvarur, India \nkevinandrews-portfolio.netlify.app/ \nlinkedin.com/in/kevinandrewsv \ngithub.com/Kevinandrewsv \nEDUCATION \nFull Stack Development \nGUVI Geek Network, IITM Research Park \n05/2023 - 05/2024,  \nChennai \nBachelor of Engineering, Mechanical \nJayam college of Engineering and\nTechnology \n05/2013 - 04/2017,  \nDharmapuri \nPERSONAL PROJECTS \nDoctor Appointment Booking Website\n (05/2024 - 06/2024) \nTechnologies used : React.js, Nodejs / Express.js, MongoDB on\nAWS \nDescription: Developed a web application for booking,\nm

In [ ]:
https://jobs.six-group.com/job/Warsaw-%28Senior%29-FrontFull-stack-Developer/1193235301/

In [21]:
final_state

{'messages': [],
 'resume_content': 'KEVIN ANDREWS, FULL STACK\nDEVELOPER\nContact Information: kevinandrews001@gmail.com | +91\n99405 53302 | Thiruvarur, India | Portfolio | LinkedIn | \nGitHub\nSummary\nHighly Motivated Full Stack Developer with expertise in front-end\nand back-end technologies.\nFocus on building efficient, scalable, and user-friendly web\napplications.\nSkilled in problem-solving, clean code, and continuous learning,\nwith a strong attention to detail and a collaborative, solution-driven\napproach.\nTechnical Skills\nFront-end: HTML, CSS, JavaScript, ReactJS, Redux, Bootstrap, Tailwind\nCSS\nBack-end: NodeJS, ExpressJS, MongoDB, MySQL\nVersion Control: Git, GitHub\nCloud: AWS\nAuthentication: OAuth, JWT\nDevelopment: API Development, Database Design, CI/CD, Error\nHandling, Performance Optimization\nSoft Skills: Collaboration, Adaptability\nProjects\n1. Doctor Appointment Booking Website (May2024 -\nJune2024)\nTechnologies Used: React.js, Node.js/Express.js, MongoD

In [3]:
def pdf_convector(paths):
    import markdown
    import pdfkit
    import re
    import os
    
    # Check if the 'paths' variable is None or not a valid path
    if not paths or not isinstance(paths, str) or not os.path.isfile(paths):
        raise ValueError(f"Invalid file path: {paths}")
    
    # Step 1: Read Markdown content
    with open(paths, "r", encoding="utf-8") as f:
        md_text = f.read()
    
    # Step 2: Extract contact info from markdown using regex
    contact_section_match = re.search(r'### Contact Information(.*?)###', md_text, re.DOTALL)
    contact_md = contact_section_match.group(1).strip() if contact_section_match else ""
    
    # Step 3: Parse contact items
    contact_items = dict()
    for line in contact_md.splitlines():
        match = re.match(r'-\s*(\w+):\s*(.+)', line)
        if match:
            key, value = match.groups()
            contact_items[key.lower()] = value.strip()
    
    # Step 4: Build dynamic contact info line
    contact_info_html = '<div class="contact-info">\n'
    if "email" in contact_items:
        contact_info_html += f'<strong>📧</strong> <a href="mailto:{contact_items["email"]}">{contact_items["email"]}</a> | '
    if "phone" in contact_items:
        phone = contact_items["phone"].replace(" ", "")
        contact_info_html += f'<strong>📞</strong> <a href="tel:+91{phone}">+91 {contact_items["phone"]}</a> | '
    if "location" in contact_items:
        contact_info_html += f'<strong>📍</strong> {contact_items["location"]} | '
    if "portfolio" in contact_items:
        contact_info_html += f'<strong>🌐</strong> <a href="{contact_items["portfolio"]}">Portfolio</a> | '
    if "linkedin" in contact_items:
        contact_info_html += f'<strong>🔗</strong> <a href="{contact_items["linkedin"]}">LinkedIn</a> | '
    if "github" in contact_items:
        contact_info_html += f'<strong>💻</strong> <a href="{contact_items["github"]}">GitHub</a>'
    contact_info_html += '\n</div><hr>\n'
    
    # Step 5: Convert Markdown to HTML
    html = markdown.markdown(md_text, extensions=["extra", "smarty"])
    
    # Step 6: Insert contact info before rendered HTML
    full_html = contact_info_html + html
    
    # Step 7: Style and wrap
    styled_html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <style>
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            margin: 40px auto;
            max-width: 850px;
            line-height: 1.6;
            color: #333;
            background-color: #fff;
        }}
        h1 {{
            color: #0a2c58;
            font-size: 2em;
            border-bottom: 2px solid #0a2c58;
            padding-bottom: 6px;
            margin-bottom: 10px;
        }}
        h2 {{
            color: #0a2c58;
            font-size: 1.5em;
            border-bottom: 1px solid #ccc;
            padding-bottom: 4px;
            margin-top: 30px;
        }}
        h3 {{
            color: #0a2c58;
            font-size: 1.2em;
            margin-top: 20px;
        }}
        p {{
            margin-bottom: 10px;
        }}
        ul {{
            padding-left: 20px;
            list-style-type: disc;
        }}
        li {{
            margin-bottom: 8px;
        }}
        a {{
            color: #0645ad;
            text-decoration: none;
        }}
        a:hover {{
            text-decoration: underline;
        }}
        section {{
            margin-bottom: 30px;
        }}
        .contact-info {{
            font-size: 0.95em;
            color: #0a2c58;
            margin-bottom: 20px;
        }}
        hr {{
            margin: 20px 0;
            border: none;
            border-top: 1px solid #ccc;
        }}
        </style>
    </head>
    <body>
    {full_html}
    </body>
    </html>
    """
    
    # Step 8: Save to HTML file
    with open("resume_temp.html", "w", encoding="utf-8") as f:
        f.write(styled_html)
    
    # Step 9: Convert to PDF
    path_to_wkhtmltopdf = r'D:\Help_project_list\resume_builder\reume_builder_checking\ai-resume-creator\wkhtmltox\bin\wkhtmltopdf.exe'
    config = pdfkit.configuration(wkhtmltopdf=path_to_wkhtmltopdf)
    pdfkit.from_file("resume_temp.html", "Kevin_Andrews_Resume.pdf", configuration=config)


## code form the claude

In [12]:
# define the state 

from typing import Annotated
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage


class Store_message(TypedDict):
    """State to store messages."""

    messages: Annotated[list[AnyMessage], add_messages]
    resume_content: str = ""
    job_description: str =""
    url:str=""
    md_formated: str =""
    score: float = 0.0
    valid: bool = True

In [ ]:
# this one for ai workflow for create the resume content

#gpt code 

# === Imports ===
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END, START

import requests
from bs4 import BeautifulSoup

# === Define State ===
class StoreMessage(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    resume_content: str
    job_description: str
    url: str
    md_formated: str
    score: float
    valid: bool

# === Import your LLM instance ===
# Define or import your LLM here (like from LangChain)
# For example:
paths = r"c:\Users\basilahamed.h\Downloads\sample_1.pdf"
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="gemma2-9b-it",
    temperature=0,
    max_tokens=None,
    timeout=60,
    groq_api_key="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",
    # other params...
)

# === Define Nodes ===
def load_pdf(state: StoreMessage) -> StoreMessage:
    from langchain_community.document_loaders import PyMuPDFLoader
    path = paths
    # print("//// pdf path ")
    # print(path)
    # print("//// pdf path ")
    loader = PyMuPDFLoader(path)
    docs = loader.load()
    for doc in docs:
        # print(doc.page_content)
        state["resume_content"] += doc.page_content
    return state

def get_job_description(state: StoreMessage) -> StoreMessage:
    response = requests.get(state["url"])
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        job_description = soup.get_text(separator="\n", strip=True)
        state["job_description"] = job_description
        return state
    else:
        raise Exception(f"Failed to fetch URL: {response.status_code}")

def llm_with_score(state: StoreMessage) -> StoreMessage:
    sys_msg = SystemMessage(content="You are an expert resume evaluator. Assess the resume's match to the job description and return a score from 0 to 100.")
    user_msg = HumanMessage(content=f"""
{state['resume_content']}

Job Description:
{state['job_description']}

Give the match score out of 100. Only return the number.
""")
    result = llm.invoke([sys_msg, user_msg])
    try:
        state["score"] = float(result.content.strip())
    except ValueError:
        state["score"] = 0.0
    return state

def create_resume(state: StoreMessage) -> StoreMessage:
    sys_msg = SystemMessage(content="""
   You are a professional resume formatter specialized in creating ATS-optimized resumes for technology companies, specifically for Amazon.

Your task is to take raw resume content from the user and convert it into a professionally formatted Markdown document that will be converted into a PDF. This resume must be fully compatible with Amazon's ATS (Applicant Tracking System) and follow all Amazon-specific resume formatting and optimization standards.

Strictly follow the formatting and logic rules below:

---

### FORMAT STRUCTURE

1. Use a clean, single-column layout with standard ATS-friendly section names:
   - Summary
   - Technical Skills
   - Experience (**only include this section if experience content is provided**)
   - Education
   - Projects
   - Certifications
   - Additional Information

2. Use Markdown structure:
   - `#` for the full name (top of document)
   - `##` for major section headings (e.g., Summary, Skills, Education)
   - `###` for job titles, project names, and degrees

3. Use a horizontal line (`---`) only to separate major sections.

---

### CONTENT RULES

- Always prioritize Amazon’s values and hiring criteria:
   - Use Amazon's Leadership Principles when relevant (e.g., customer obsession, ownership, invent and simplify, learn and be curious).
   - Highlight quantifiable achievements (e.g., increased efficiency by 25%, reduced costs by $10K).
   - Prioritize Amazon-relevant technologies (e.g., AWS, Python, Java, DynamoDB).

- Do not fabricate any content. If the user does not provide work experience, omit the “Experience” section entirely.

- If experience is present, use the following substructure:
  - `### [Job Title]`
    `**[Company Name]** | [Start Date] – [End Date]`
    - [Achievement or responsibility #1]
    - [Achievement or responsibility #2]
    - [Quantified result or technology used]

---

### STYLING RULES

- Use bullet points (`-`) for achievements and skills.
- Use `**bold**` for:
  - Company names
  - Job titles
  - Skills categories
- Do not use tables, columns, graphics, or any non-ATS-friendly elements.
- Do not use headers, footers, or special symbols.

---

### TECHNICAL NOTES

- Ensure all dates follow a consistent format: MM/YYYY or Month YYYY.
- Only use standard ASCII characters.
- Ensure section order is exactly as follows:
  1. Summary
  2. Technical Skills
  3. [Experience – only if provided]
  4. Education
  5. Projects
  6. Certifications
  7. Additional Information

---

### OUTPUT FORMAT

Provide the final resume in **clean, correctly structured Markdown** using the above rules. Do not output any commentary or notes — only the resume Markdown content.

DO NOT include the "Experience" section in the output if no experience content was provided by the user.

Company domain context: **Amazon (amazon.com)**. Optimize all phrasing, structuring, and terminology for Amazon’s tech hiring needs.


    """)
    user_msg = HumanMessage(content=f"""
My resume:
{state["resume_content"]}

Job description:
{state["job_description"]}
""")
    final = llm.invoke([sys_msg, user_msg])
    state["md_formated"] = final.content
    return state

def llm_review(state: StoreMessage) -> StoreMessage:
    state["valid"] = False
    return state

# === Graph Definition ===
graph = StateGraph(StoreMessage)

graph.add_node("load_pdf", load_pdf)
graph.add_node("get_job_description", get_job_description)
graph.add_node("llm_with_score", llm_with_score)
graph.add_node("create_resume", create_resume)
graph.add_node("llm_review", llm_review)

graph.set_entry_point("load_pdf")

graph.add_edge("load_pdf", "get_job_description")
graph.add_edge("get_job_description", "llm_with_score")

def score_check(state: StoreMessage) -> str:
    return "create_resume" if state["score"] > 70 else "llm_review"

graph.add_conditional_edges("llm_with_score", score_check)

graph.add_edge("create_resume", END)
graph.add_edge("llm_review", END)

workflow = graph.compile()


In [38]:
#prompt optimizing

# this one for ai workflow for create the resume content

#gpt code 

# === Imports ===
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END, START

import requests
from bs4 import BeautifulSoup

# === Define State ===
class StoreMessage(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    resume_content: str
    job_description: str
    url: str
    md_formated: str
    score: float
    valid: bool

# === Import your LLM instance ===
# Define or import your LLM here (like from LangChain)
# For example:
paths = r"c:\Users\basilahamed.h\Downloads\sample_1.pdf"
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="gemma2-9b-it",
    temperature=0,
    max_tokens=None,
    timeout=60,
    groq_api_key="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",
    # other params...
)

# === Define Nodes ===
def load_pdf(state: StoreMessage) -> StoreMessage:
    from langchain_community.document_loaders import PyMuPDFLoader
    path = paths
    # print("//// pdf path ")
    # print(path)
    # print("//// pdf path ")
    loader = PyMuPDFLoader(path)
    docs = loader.load()
    for doc in docs:
        # print(doc.page_content)
        state["resume_content"] += doc.page_content
    return state

def get_job_description(state: StoreMessage) -> StoreMessage:
    response = requests.get(state["url"])
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        job_description = soup.get_text(separator="\n", strip=True)
        state["job_description"] = job_description
        return state
    else:
        raise Exception(f"Failed to fetch URL: {response.status_code}")

def llm_with_score(state: StoreMessage) -> StoreMessage:
    sys_msg = SystemMessage(content="You are an expert resume evaluator. Assess the resume's match to the job description and return a score from 0 to 100.")
    user_msg = HumanMessage(content=f"""
{state['resume_content']}

Job Description:
{state['job_description']}

Give the match score out of 100. Only return the number.
""")
    result = llm.invoke([sys_msg, user_msg])
    try:
        state["score"] = float(result.content.strip())
    except ValueError:
        state["score"] = 0.0
    return state

def create_resume(state: StoreMessage) -> StoreMessage:
    sys_msg = SystemMessage(content="""
    You are an expert Large Language Model acting as a professional resume formatter for the domain **Amazon (amazon.com)**. Your role is to take raw resume content provided by the user and transform it into a clean, ATS-optimized Markdown document. This document will be used to generate a PDF resume suitable for submission to Amazon's applicant tracking system (ATS).

Your formatting must fully comply with Amazon’s resume screening requirements, including:

---

### 🔹 SECTION ORDER & STRUCTURE

1. **Name & Contact Information**  
   Start with the user’s full name using a level-1 Markdown heading (`#`).  
   Directly below, include a single line of contact information with:
   - Location (City, State or Country)
   - Email (as a clickable mailto link)
   - Phone number
   - LinkedIn (as a clickable hyperlink)

   **Markdown Example:**
2. ** domian of the resume**
    -analyst the resume and give if the resume is have any domin if present mean give the domin name else give the domain name based on the resume content and job description.

2. **Summary** (`## Summary`)  
3 short 2–4 sentence professional summary tailored to Amazon’s values (e.g., customer obsession, ownership, innovation). Emphasize relevant roles, skills, and Amazon-compatible experience.

3. **Technical Skills** (`## Technical Skills`)  
Use categorized bullet points. Format each category in bold:


4. **Experience** (`## Professional Experience`)  
**Only include this section if experience data is present.**  
For each role:
- Use level-3 headings for job titles: `### [Job Title]`
- Format company name in bold and include dates:
  ```
  **[Company Name]** | MM/YYYY – MM/YYYY
  ```
- Use bullet points for accomplishments, focusing on:
  - Quantifiable results
  - Amazon Leadership Principles
  - Relevant technologies

5. **Education** (`## Education`)  
For each degree:
- Use level-3 headings for degree name
- University name in bold with dates
- Include relevant honors or coursework (optional)

6. **Projects** (`## Projects`)  
- Use level-3 headings for each project name
- Describe the tech stack used and key contributions
- Focus on impact or deployment

7. **Certifications** (`## Certifications`)  
- List technical or leadership certifications relevant to Amazon

8. **Additional Information** (`## Additional Information`)  
- Languages, open-source contributions, hobbies (if applicable)

---

### 🔹 CONTENT GUIDELINES

- Use **Amazon’s Leadership Principles** when appropriate:  
*Customer Obsession, Ownership, Invent and Simplify, Learn and Be Curious, Deliver Results*  
- Quantify all results: `% improvements`, `$ saved`, `time reduced`
- Mention Amazon-aligned tech (AWS, DynamoDB, Java, Python, etc.)

---

### 🔹 STYLING RULES

- Use `**bold**` for:
- Job titles
- Company names
- Skills categories
- Use `-` for bullet points (no checkboxes or asterisks)
- Use consistent date formatting: MM/YYYY or Month YYYY
- Do **not** use:
- Tables
- Multiple columns
- Images, logos, or headers/footers

---

### 🔹 TECHNICAL CONDITIONS

- Resume **must be valid Markdown**
- Use only standard ASCII characters
- Do **not** output the "Experience" section if no work history is provided
- Output only the final resume — no additional comments, headings, or text outside of the resume

---

### 🔹 DOMAIN CONTEXT

This resume is for submission to **Amazon** — a global technology company (domain: `amazon.com`).  
Prioritize formatting and phrasing optimized for Amazon’s ATS and hiring process.

---

### 🔹 OUTPUT FORMAT (Markdown Example):

```markdown
# Jane Doe  
San Francisco, CA | [jane.doe@example.com](mailto:jane.doe@example.com) | (555) 123-4567 | [linkedin.com/in/janedoe](https://linkedin.com/in/janedoe)

## Summary
Innovative software engineer with 5+ years of experience delivering scalable cloud solutions. Known for strong ownership and customer obsession. Proficient in AWS, Python, and microservices. Adept at solving complex backend problems with measurable impact.

## Technical Skills
- **Languages**: Python, Java, Go
- **Cloud**: AWS (EC2, S3, Lambda), Docker, Kubernetes
- **CI/CD**: Jenkins, GitHub Actions
- **Databases**: PostgreSQL, DynamoDB

## Professional Experience

### Backend Engineer  
**XYZ Corp** | 06/2019 – Present  
- Spearheaded migration to AWS Lambda, reducing infrastructure cost by 30%  
- Led team of 4 engineers to build REST APIs serving over 1M monthly users  
- Practiced ownership by redesigning critical failure-prone components, improving uptime by 15%

## Education

### B.S. in Computer Science  
**University of Washington** | 2014 – 2018  
- GPA: 3.8/4.0  
- Relevant coursework: Distributed Systems, Data Structures

## Projects

### Inventory Management System  
- Built a scalable system using Python and DynamoDB  
- Reduced inventory discrepancies by 40% after deployment

## Certifications
- AWS Certified Solutions Architect – Associate  
- Certified Kubernetes Administrator (CKA)

## Additional Information
- Fluent in Spanish and English  
- Contributor to open-source project: FastAPI



    """)
    user_msg = HumanMessage(content=f"""
My resume:
{state["resume_content"]}

Job description:
{state["job_description"]}
""")
    final = llm.invoke([sys_msg, user_msg])
    state["md_formated"] = final.content
    return state

def llm_review(state: StoreMessage) -> StoreMessage:
    state["valid"] = False
    return state

# === Graph Definition ===
graph = StateGraph(StoreMessage)

graph.add_node("load_pdf", load_pdf)
graph.add_node("get_job_description", get_job_description)
graph.add_node("llm_with_score", llm_with_score)
graph.add_node("create_resume", create_resume)
graph.add_node("llm_review", llm_review)

graph.set_entry_point("load_pdf")

graph.add_edge("load_pdf", "get_job_description")
graph.add_edge("get_job_description", "llm_with_score")

def score_check(state: StoreMessage) -> str:
    return "create_resume" if state["score"] > 70 else "llm_review"

graph.add_conditional_edges("llm_with_score", score_check)

graph.add_edge("create_resume", END)
graph.add_edge("llm_review", END)

workflow = graph.compile()


In [ ]:
# domin maine specification

#prompt optimizing

# this one for ai workflow for create the resume content

#gpt code 

# === Imports ===
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END, START

import requests
from bs4 import BeautifulSoup

# === Define State ===
class StoreMessage(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    resume_content: str
    job_description: str
    url: str
    md_formated: str
    score: float
    valid: bool

# === Import your LLM instance ===
# Define or import your LLM here (like from LangChain)
# For example:
paths = r"c:\Users\basilahamed.h\Downloads\sample_1.pdf"
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="gemma2-9b-it",
    temperature=0,
    max_tokens=None,
    timeout=60,
    groq_api_key="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",
    # other params...
)

# === Define Nodes ===
def load_pdf(state: StoreMessage) -> StoreMessage:
    from langchain_community.document_loaders import PyMuPDFLoader
    path = paths
    # print("//// pdf path ")
    # print(path)
    # print("//// pdf path ")
    loader = PyMuPDFLoader(path)
    docs = loader.load()
    for doc in docs:
        # print(doc.page_content)
        state["resume_content"] += doc.page_content
    return state

def get_job_description(state: StoreMessage) -> StoreMessage:
    response = requests.get(state["url"])
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        job_description = soup.get_text(separator="\n", strip=True)
        state["job_description"] = job_description
        return state
    else:
        raise Exception(f"Failed to fetch URL: {response.status_code}")

def llm_with_score(state: StoreMessage) -> StoreMessage:
    sys_msg = SystemMessage(content="You are an expert resume evaluator. Assess the resume's match to the job description and return a score from 0 to 100.")
    user_msg = HumanMessage(content=f"""
{state['resume_content']}

Job Description:
{state['job_description']}

Give the match score out of 100. Only return the number.
""")
    result = llm.invoke([sys_msg, user_msg])
    try:
        state["score"] = float(result.content.strip())
    except ValueError:
        state["score"] = 0.0
    return state

def create_resume(state: StoreMessage) -> StoreMessage:
    sys_msg = SystemMessage(content="""
You are an expert Large Language Model acting as a professional resume formatter, specializing in job applications. Your task is to take the user’s raw resume content and convert it into a professionally structured, ATS-optimized Markdown resume that follows this exact format and section order:

🔹 OUTPUT STRUCTURE (Always in This Order)

Full Name
Contact Information
Single line format:
Location | Email | Phone | LinkedIn

All links must be valid and clickable in Markdown format.

Domain Name (Heading Format)
Detect the most relevant professional domain (e.g., Full Stack Developer, Cybersecurity Analyst, Data Scientist)
Print the domain as a second-level heading (##) directly under the contact info, without the label "Domain:".

Example:

markdown
Copy
Edit
## Full Stack Developer
Summary
2–4 concise sentences
Brief intro, experience level, technical focus
Tie to relevant professional skills and leadership qualities.

Experience
(Include only if experience is provided)
For each job:

[Job Title]
[Company Name] | MM/YYYY – MM/YYYY

[Bullet describing quantifiable achievement]

[Bullet showing impact, leadership, or relevant results]

Technical Skills
Grouped by categories with bold labels:

Languages: Python, Java, JavaScript

Cloud: AWS, Azure

Tools: Docker, GitHub Actions

Projects
For each project:

[Project Name]
[What it is + tech used]

[Impact or measurable outcome]

Education
[Degree Name]
[University Name] | MM/YYYY – MM/YYYY

[Optional GPA or coursework]

Certifications
[Certification Name]

[Certification Name]

Additional Information
Languages spoken, open-source contributions, hobbies, awards, or other relevant items.

🔹 DOMAIN DETECTION RULES
You must detect the most suitable domain automatically based on:

Skills

Experience

Projects

Keywords

Examples:

React, Node.js, MongoDB → Full Stack Developer

SIEM, threat detection, network security → Cybersecurity Analyst

Pandas, Machine Learning, Scikit-learn → Data Scientist

The domain name must be formatted as:

markdown
Copy
Edit
## Full Stack Developer
🔹 FORMATTING RULES

Use bold for job titles, company names, skill group headers, and domain.

Use - (dash) for bullet points.

Do not use tables, checkboxes, images, or non-standard Markdown.

Use consistent date format: MM/YYYY or Month YYYY.

Use # for name, ## for sections, and ### for jobs/projects/education entries.

All hyperlinks (email, LinkedIn) must be in proper Markdown link format and functional.

Do not add any extra commentary or instructions in the output.

Resume must be clean Markdown, 100% ATS-compliant.

🔹 TECHNICAL CONSTRAINTS

Return only the final resume in Markdown — no explanations or headers.

Include all content from the user’s original resume (no omissions).

Do not include "Experience" section if the user hasn’t provided it.

The resume must align with relevant hiring values and be optimized for ATS scanning.



...
    """)
    user_msg = HumanMessage(content=f"""
My resume:
{state["resume_content"]}

Job description:
{state["job_description"]}
""")
    final = llm.invoke([sys_msg, user_msg])
    state["md_formated"] = final.content
    return state

def llm_review(state: StoreMessage) -> StoreMessage:
    state["valid"] = False
    return state

# === Graph Definition ===
graph = StateGraph(StoreMessage)

graph.add_node("load_pdf", load_pdf)
graph.add_node("get_job_description", get_job_description)
graph.add_node("llm_with_score", llm_with_score)
graph.add_node("create_resume", create_resume)
graph.add_node("llm_review", llm_review)

graph.set_entry_point("load_pdf")

graph.add_edge("load_pdf", "get_job_description")
graph.add_edge("get_job_description", "llm_with_score")

def score_check(state: StoreMessage) -> str:
    return "create_resume" if state["score"] > 70 else "llm_review"

graph.add_conditional_edges("llm_with_score", score_check)

graph.add_edge("create_resume", END)
graph.add_edge("llm_review", END)

workflow = graph.compile()


In [50]:
import argparse
import markdown
import os
import subprocess
import tempfile

# Load the ATS-friendly CSS for Amazon resumes
AMAZON_ATS_CSS = """
/* ATS-Friendly Resume CSS for Amazon Applications */

@page {
    margin: 0.5in;
    size: letter portrait;
}

body {
    font-family: 'Calibri', 'Arial', sans-serif;
    font-size: 11pt;
    line-height: 1.4;
    color: #333333;
    margin: 0;
    padding: 0;
}

.container {
    max-width: 8.5in;
    margin: 0 auto;
    padding: 0.25in;
}

/* Name and Header */
h1 {
    font-size: 18pt;
    font-weight: bold;
    color: #232f3e; /* Amazon blue */
    margin-bottom: 0.1in;
    text-align: center;
}

/* Contact Information */
p:first-of-type {
    text-align: center;
    margin-bottom: 0.3in;
    font-size: 10pt;
}

/* Section Headings */
h2 {
    font-size: 14pt;
    font-weight: bold;
    color: #232f3e; /* Amazon blue */
    margin-top: 0.3in;
    margin-bottom: 0.1in;
    padding-bottom: 0.05in;
    border-bottom: 1px solid #888;
}

/* Sub-headings (job titles, education degrees) */
h3 {
    font-size: 11pt;
    font-weight: bold;
    margin-top: 0.2in;
    margin-bottom: 0.05in;
}

/* Job details */
h3 + p {
    margin-top: -0.05in;
    margin-bottom: 0.1in;
}

/* Lists */
ul {
    margin-top: 0.05in;
    margin-bottom: 0.15in;
    padding-left: 0.2in;
}

li {
    margin-bottom: 0.05in;
    list-style-type: disc;
}

/* Emphasis for company names and skills */
strong {
    font-weight: bold;
}

/* Ensure proper text wrapping */
p, li {
    word-wrap: break-word;
}
"""

# ATS-friendly HTML template for Amazon resumes
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{name} - Amazon Resume</title>
    <style>
        {css}
    </style>
</head>
<body>
    <div class="container">
        {content}
    </div>
</body>
</html>
"""

def extract_name(content):
    """Attempt to extract the name from the markdown content"""
    lines = content.split('\n')
    for line in lines:
        if line.strip().startswith('# '):
            return line.strip()[2:].strip()
    return "Professional Resume"

def convert_md_to_html(md_content):
    """Convert markdown content to HTML with Amazon ATS-friendly styling"""
    html_content = markdown.markdown(md_content)
    name = extract_name(md_content)
    return HTML_TEMPLATE.format(name=name, content=html_content, css=AMAZON_ATS_CSS)

def convert_html_to_pdf(html_content, output_path):
    """Convert HTML content to PDF using pdfkit with local wkhtmltopdf."""
    # Path to your wkhtmltopdf executable
    path_to_wkhtmltopdf = r'D:\Help_project_list\resume_builder\reume_builder_checking\ai-resume-creator\wkhtmltox\bin\wkhtmltopdf.exe'
    config = pdfkit.configuration(wkhtmltopdf=path_to_wkhtmltopdf)

    options = {
        'enable-local-file-access': '',
        'print-media-type': '',
        'no-background': '',
        'margin-top': '12mm',
        'margin-bottom': '12mm',
        'margin-left': '12mm',
        'margin-right': '12mm',
        'page-size': 'Letter',
        'dpi': 300,
        'disable-smart-shrinking': ''
    }

    try:
        pdfkit.from_string(html_content, output_path, configuration=config, options=options)
        print(f"ATS-friendly Amazon resume PDF successfully created at: {output_path}")
    except Exception as e:
        print(f"Error creating PDF: {e}")

    finally:
        # Clean up the temporary file
        if os.path.exists(temp_html_path):
            os.unlink(temp_html_path)

    


In [ ]:
import argparse
import markdown
import os
import subprocess
import tempfile
import pdfkit


path_to_wkhtmltopdf = r'D:\Help_project_list\resume_builder\reume_builder_checking\ai-resume-creator\wkhtmltox\bin\wkhtmltopdf.exe'
config = pdfkit.configuration(wkhtmltopdf=path_to_wkhtmltopdf)

# Load the ATS-friendly CSS for Amazon resumes
AMAZON_ATS_CSS = """
/* ATS-Friendly Resume CSS for Amazon Applications */

@page {
    margin: 0.5in;
    size: letter portrait;
}

body {
    font-family: 'Calibri', 'Arial', sans-serif;
    font-size: 11pt;
    line-height: 1.4;
    color: #333333;
    margin: 0;
    padding: 0;
}

.container {
    max-width: 8.5in;
    margin: 0 auto;
    padding: 0.25in;
}

/* Name and Header */
h1 {
    font-size: 18pt;
    font-weight: bold;
    color: #232f3e; /* Amazon blue */
    margin-bottom: 0.1in;
    text-align: center;
}

/* Contact Information */
p:first-of-type {
    text-align: center;
    margin-bottom: 0.3in;
    font-size: 10pt;
}

/* Section Headings */
h2 {
    font-size: 14pt;
    font-weight: bold;
    color: #232f3e; /* Amazon blue */
    margin-top: 0.3in;
    margin-bottom: 0.1in;
    padding-bottom: 0.05in;
    border-bottom: 1px solid #888;
}

/* Sub-headings (job titles, education degrees) */
h3 {
    font-size: 11pt;
    font-weight: bold;
    margin-top: 0.2in;
    margin-bottom: 0.05in;
}

/* Job details */
h3 + p {
    margin-top: -0.05in;
    margin-bottom: 0.1in;
}

/* Lists */
ul {
    margin-top: 0.05in;
    margin-bottom: 0.15in;
    padding-left: 0.2in;
}

li {
    margin-bottom: 0.05in;
    list-style-type: disc;
}

/* Emphasis for company names and skills */
strong {
    font-weight: bold;
}

/* Ensure proper text wrapping */
p, li {
    word-wrap: break-word;
}
"""

# ATS-friendly HTML template for Amazon resumes
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{name} - Amazon Resume</title>
    <style>
        {css}
    </style>
</head>
<body>
    <div class="container">
        {content}
    </div>
</body>
</html>
"""

def extract_name(content):
    """Attempt to extract the name from the markdown content"""
    lines = content.split('\n')
    for line in lines:
        if line.strip().startswith('# '):
            return line.strip()[2:].strip()
    return "Professional Resume"

def convert_md_to_html(md_content):
    """Convert markdown content to HTML with Amazon ATS-friendly styling"""
    html_content = markdown.markdown(md_content)
    name = extract_name(md_content)
    return HTML_TEMPLATE.format(name=name, content=html_content, css=AMAZON_ATS_CSS)

def convert_html_to_pdf(html_content, output_path):
    """Convert HTML content to PDF using wkhtmltopdf with settings for Amazon ATS"""
    with tempfile.NamedTemporaryFile(suffix='.html', delete=False) as temp:
        temp.write(html_content.encode('utf-8'))
        temp_html_path = temp.name
    
    try:
        # Run wkhtmltopdf command with optimal settings for ATS
        subprocess.run([
            'wkhtmltopdf',
            '--enable-local-file-access',
            '--print-media-type',
            '--no-background',
            '--margin-top', '12mm',
            '--margin-bottom', '12mm',
            '--margin-left', '12mm',
            '--margin-right', '12mm',
            '--page-size', 'Letter',
            '--dpi', '300',
            '--disable-smart-shrinking',
            temp_html_path,
            output_path
        ], check=True)
        print(f"ATS-friendly Amazon resume PDF successfully created at: {output_path}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"Error creating PDF: {e}")
        return False
    except FileNotFoundError:
        print("Error: wkhtmltopdf not found. Please install it first.")
        return False
    finally:
        # Clean up the temporary file
        if os.path.exists(temp_html_path):
            os.unlink(temp_html_path)

def save_markdown_to_file(md_content, filename="resume.md"):
    """Save markdown content to a file"""
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(md_content)
        return filename
    except Exception as e:
        print(f"Error saving markdown file: {e}")
        return None

def pdf_converter(markdown_path, output_path="amazon_resume.pdf"):
    """Convert markdown file to PDF"""
    try:
        with open(markdown_path, 'r', encoding='utf-8') as f:
            md_content = f.read()
        
        html_content = convert_md_to_html(md_content)
        return convert_html_to_pdf(html_content, output_path)
    except Exception as e:
        print(f"Error in PDF conversion: {e}")
        return False


In [ ]:
# import os

# def save_markdown_to_file(md_content: str, file_path: str = "optimized_resume.md") -> str:
#     # Get the absolute path of the file
#     full_path = os.path.abspath(file_path)
    
#     # Write the content to the file
#     with open(full_path, "w", encoding="utf-8") as f:
#         f.write(md_content)
    
#     print(f"✅ Markdown resume saved to: {full_path}")
#     return full_path

In [31]:
import pdfkit
final_state = workflow.invoke({
    "messages": [],
    "resume_content": "",
    "job_description": "",
    "url": "https://www.zoho.com/careers/jobdetails/?job_id=2803000614929615",
    "md_formated": "",
    "score": 0.0,
    "valid": True
})

# Save the markdown file

if len(final_state["md_formated"]) > 0:
    # final_path  = save_markdown_to_file(final_state["md_formated"])
    # save_padf = pdf_convector(final_path)
        # Save markdown to file
    path_to_wkhtmltopdf = r'D:\Help_project_list\resume_builder\reume_builder_checking\ai-resume-creator\wkhtmltox\bin\wkhtmltopdf.exe'
    config = pdfkit.configuration(wkhtmltopdf=path_to_wkhtmltopdf)
    md_file = save_markdown_to_file(final_state["md_formated"])
        
    if md_file:
        # Convert to PDF
        pdf_converter(md_file)
    else:
        print("Resume not satisfied.")

ATS-friendly Amazon resume PDF successfully created at: amazon_resume.pdf
Error in PDF conversion: name 'temp_html_path' is not defined


In [51]:
import pdfkit
final_state = workflow.invoke({
    "messages": [],
    "resume_content": "",
    "job_description": "",
    "url": "https://jobs.six-group.com/job/Warsaw-%28Senior%29-FrontFull-stack-Developer/1193235301/",
    "md_formated": "",
    "score": 0.0,
    "valid": True
})

# Save the markdown file

if len(final_state["md_formated"]) > 0:
    # final_path  = save_markdown_to_file(final_state["md_formated"])
    # save_padf = pdf_convector(final_path)
        # Save markdown to file
    # path_to_wkhtmltopdf = r'D:\Help_project_list\resume_builder\reume_builder_checking\ai-resume-creator\wkhtmltox\bin\wkhtmltopdf.exe'
    # config = pdfkit.configuration(wkhtmltopdf=path_to_wkhtmltopdf)
    md_file = save_markdown_to_file(final_state["md_formated"])
        
    if md_file:
        # Convert to PDF
        pdf_converter(md_file)
    else:
        print("Resume not satisfied.")

ATS-friendly Amazon resume PDF successfully created at: amazon_resume.pdf
Error in PDF conversion: name 'temp_html_path' is not defined


In [ ]:
!set wkhtmltopdf = 

In [29]:
!wkhtmltopdf --version

'wkhtmltopdf' is not recognized as an internal or external command,
operable program or batch file.
